In [0]:
from pyspark.sql.functions import (
    col, to_timestamp, lower, trim, regexp_replace, when,
    hour, month, dayofweek
)

# path
bronze_volume_path = "/Volumes/workspace/default/uploads/US_Accidents_March23.csv"
silver_table = "workspace.default.silver_accidents"

raw_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(bronze_volume_path)

print(f"Loaded {raw_df.count()} rows from Bronze")

In [0]:
df = raw_df

# Parse timestamps
df = df.withColumn("Start_Time", to_timestamp(col("Start_Time"))) \
       .withColumn("End_Time", to_timestamp(col("End_Time")))

# Extract time features
df = df.withColumn("hour", hour(col("Start_Time"))) \
       .withColumn("month", month(col("Start_Time"))) \
       .withColumn("day_of_week", dayofweek(col("Start_Time")))

# Cast numeric columns
df = df.withColumn("Severity", col("Severity").cast("integer")) \
       .withColumn("Start_Lat", col("Start_Lat").cast("double")) \
       .withColumn("Start_Lng", col("Start_Lng").cast("double")) \
       .withColumn("Distance(mi)", col("Distance(mi)").cast("double")) \
       .withColumn("Temperature(F)", col("Temperature(F)").cast("double")) \
       .withColumn("Visibility(mi)", col("Visibility(mi)").cast("double")) \
       .withColumn("Wind_Speed(mph)", col("Wind_Speed(mph)").cast("double"))

print("Casting done")

In [0]:
# Drop rows with nulls in critical columns
df = df.dropna(subset=[
    "Severity", "Start_Time", "Start_Lat", "Start_Lng",
    "Temperature(F)", "Visibility(mi)"
])

print(f"After dropping nulls: {df.count()} rows")

In [0]:
# Filter top 25 weather conditions
top_25 = [r[0] for r in df.groupBy("Weather_Condition").count()
          .orderBy(col("count").desc()).limit(25).collect()]
df = df.filter(col("Weather_Condition").isin(top_25))

# Standardize weather condition labels
df = df.withColumn("Weather_Condition", lower(trim(
    regexp_replace(col("Weather_Condition"), r'[/\\-]', ' '))))
df = df.withColumn("Weather_Condition",
    regexp_replace(col("Weather_Condition"), r'\s+', ' '))

df = df.withColumn("Weather_Condition",
    when(col("Weather_Condition").rlike('light rain with thunder|thunder|t storm'), 'Rainy_Thunder')
    .when(col("Weather_Condition").rlike('light rain|light drizzle|rain'), 'Rain_Drizzle')
    .when(col("Weather_Condition").rlike('light snow|snow'), 'Snow')
    .when(col("Weather_Condition").rlike('fair|clear'), 'Clear_Fair')
    .when(col("Weather_Condition").rlike('mostly cloudy windy|cloudy windy'), 'Cloudy_Windy')
    .when(col("Weather_Condition").rlike('mostly cloudy|cloudy|partly cloudy|overcast|scattered clouds'), 'Cloudy')
    .otherwise(col("Weather_Condition"))
)

print("Weather conditions standardized")

In [0]:
# Remove outliers using IQR
def get_iqr_bounds(df, column, lower_multiplier=1.5, upper_multiplier=1.5):
    quantiles = df.select(col(column).cast("float").alias("temp_col")) \
                  .approxQuantile("temp_col", [0.25, 0.75], 0.05)
    iqr = quantiles[1] - quantiles[0]
    return quantiles[0] - (lower_multiplier * iqr), quantiles[1] + (upper_multiplier * iqr)

wind_lower, wind_upper = get_iqr_bounds(df, "Wind_Speed(mph)")
temp_lower, temp_upper = get_iqr_bounds(df, "Temperature(F)",
                                         lower_multiplier=3.0, upper_multiplier=1.5)

df = df.filter(
    (col("Wind_Speed(mph)") >= wind_lower) & (col("Wind_Speed(mph)") <= wind_upper) &
    (col("Temperature(F)").cast("float") >= temp_lower) &
    (col("Temperature(F)").cast("float") <= temp_upper)
)

print(f"After outlier removal: {df.count()} rows")

In [0]:
# Remove duplicates and trim string columns
df = df.dropDuplicates()
for c in ["State", "Street", "County", "City"]:
    df = df.withColumn(c, trim(col(c)))

# Rename columns to remove special characters
df = df.withColumnRenamed("Distance(mi)", "Distance_mi") \
       .withColumnRenamed("Temperature(F)", "Temperature_F") \
       .withColumnRenamed("Visibility(mi)", "Visibility_mi") \
       .withColumnRenamed("Wind_Speed(mph)", "Wind_Speed_mph") \
       .withColumnRenamed("Wind_Chill(F)", "Wind_Chill_F") \
       .withColumnRenamed("Humidity(%)", "Humidity_pct") \
       .withColumnRenamed("Pressure(in)", "Pressure_in") \
       .withColumnRenamed("Precipitation(in)", "Precipitation_in")

print("Renaming done")
df.printSchema()


In [0]:
# Save as Silver Delta Table
df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
print(f"Silver layer done! Rows: {df.count()}")
df.show(5)

In [0]:
# Verify
spark.sql("SELECT * FROM workspace.default.silver_accidents LIMIT 5").show()